<a href="https://colab.research.google.com/github/doshimihir07/agentic_ai/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Installing and Importing Necessary Libraries

In [ ]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.73 --force-reinstall --no-cache-dir -q

**Note**:
- After running the above cell, kindly restart the runtime and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
!pip install langchain==0.3.27 \
            langchain-community==0.3.27 \
            torch==2.6.0 \
            sentence_transformers==5.1.1 \
            faiss-cpu==1.12.0 \
            pypdf==6.1.3 \
            huggingface_hub==0.35.3 \
            torchvision==0.21.0

**Note**:
- After running the above cell, kindly restart the runtime and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
from langchain.chains import RetrievalQA
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from huggingface_hub import hf_hub_download
from langchain.llms import LlamaCpp
# from langchain.llms import Ollama

## Load the data

In [ ]:
#  ADD PATH
loader = TextLoader('AAPL-MDA.txt')
data = loader.load()

## Split the Extracted Data into Text Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

text_chunks = text_splitter.split_documents(data)

In [ ]:
len(text_chunks)

In [ ]:
#get the third chunk
text_chunks[3].page_content

In [ ]:
text_chunks[2].page_content

In [ ]:
len(text_chunks[22].page_content)

## Load the embedding model

In [ ]:
# https://huggingface.co/spaces/mteb/leaderboard
embeddings = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")

## Create Embeddings for each of the Text Chunk

In [ ]:
vector_store = FAISS.from_documents(text_chunks, embedding=embeddings)

In [ ]:
text_chunks[0].page_content

In [ ]:
vector_store.index.reconstruct(0)

## Load the LLM

In [ ]:
# Define the model repository and the file name from HuggingFace Hub
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

# Download the model file from HuggingFace Hub
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [ ]:
# Initialize the LlamaCpp model with configuration
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.01,
    top_p=0.95,
    verbose=False,
    n_ctx=4096,
    n_batch=1024,
    n_gpu_layers=43,
)

In [ ]:
# https://ollama.com/library
# llm = Ollama(model='llama3.1:70b')

## Build the chain

In [ ]:
agent_colab = RetrievalQA.from_chain_type(llm=llm,verbose=True,chain_type="stuff",
                                          retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
                                          chain_type_kwargs={"verbose": True})

## Run the query

In [ ]:
# The Company performs a detailed review of inventory each fiscal quarter that considers multiple factors including demand forecasts, product life cycle status, product development plans, current sales levels, and component cost trends.

query ="How often does the company review inventory, and what is considered in this inventory calculation?"

In [ ]:
output_colab = agent_colab.run(query)

In [ ]:
print(output_colab)